# 04 — Automated Reporting Pipeline

End-to-end enterprise pipeline:
1. ETL: normalize raw EHR data
2. Risk scoring: score today's patient population
3. Drug safety: flag dangerous combinations
4. Reports: generate HTML + Excel outputs

This is the pattern for a nightly automated health informatics job.

In [ ]:
from pathlib import Path
from pyhealth_enterprise.config import settings
from pyhealth_enterprise.pipelines.ehr_etl import EHRETL
from pyhealth_enterprise.pipelines.drug_safety_checker import DrugSafetyChecker
from pyhealth_enterprise.pipelines.report_generator import ReportGenerator
from pyhealth_enterprise.datasets.synthetic import SyntheticEHRDataset
from pyhealth_enterprise.tasks.readmission import setup_readmission_task
from pyhealth_enterprise.models.registry import ModelName, get_model
from pyhealth_enterprise.pipelines.batch_risk_scorer import BatchRiskScorer
from pyhealth.tasks import readmission_prediction_mimic3_fn

OUTPUT_DIR = settings.PROJECT_ROOT / 'data' / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Step 1: ETL
print('Step 1: ETL...')
etl = EHRETL(source_dir=settings.SYNTHETIC_DATA_PATH, output_dir=OUTPUT_DIR / 'normalized')
etl.run()

In [ ]:
# Step 2: Risk scoring
print('Step 2: Risk scoring...')
ds = SyntheticEHRDataset(); ds.load()
train_l, val_l, test_l = setup_readmission_task(ds.dataset)
task_dataset = ds.dataset.set_task(readmission_prediction_mimic3_fn)
model = get_model(ModelName.RETAIN, task_dataset, ['conditions','drugs'], 'readmission')
scorer = BatchRiskScorer(model, 'retain_nightly')
scorer.train(train_l, val_l, epochs=20)
risk_df = scorer.score_batch(test_l)
print(f'  Scored {len(risk_df)} patients')

In [ ]:
# Step 3: Drug safety for high-risk patients
print('Step 3: Drug safety check...')
checker = DrugSafetyChecker()
high_risk_patients = risk_df[risk_df['risk_label'] == 'high']
print(f'  High-risk patients: {len(high_risk_patients)}')

In [ ]:
# Step 4: Reports
print('Step 4: Generating reports...')
report_gen = ReportGenerator()
report_gen.generate_risk_summary(risk_df, OUTPUT_DIR / 'nightly_risk_summary.html')
report_gen.generate_risk_excel(risk_df, OUTPUT_DIR / 'nightly_risk_scores.xlsx')
print(f'  Reports written to {OUTPUT_DIR}')
print('Pipeline complete.')